# Tutorial 1: Basic Molecule Generation

This tutorial demonstrates how to train and use the E3 Equivariant Diffusion Model for basic 3D molecule generation.

**Time to complete**: 15-20 minutes (with pre-trained model) or 2-3 hours (training from scratch)

**What you'll learn**:
- Load and prepare QM9 dataset
- Train a basic diffusion model
- Generate new molecules
- Evaluate molecule quality

**Prerequisites**: Basic Python, PyTorch knowledge

## Setup and Imports

In [ ]:
import sys
import os

# Add repository root to path
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import argparse

# Import project modules
from qm9 import dataset
from qm9.models import get_model, get_optim
from configs.datasets_config import get_dataset_info
from equivariant_diffusion import utils as flow_utils

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 1. Data Loading

We'll use a small subset of QM9 for quick training. QM9 contains ~134k small organic molecules.

In [ ]:
# Setup arguments
args = argparse.Namespace()

# Dataset settings
args.dataset = 'qm9'
args.batch_size = 32
args.num_workers = 2
args.filter_n_atoms = None
args.datadir = 'data'
args.remove_h = False
args.include_charges = True

# For quick demo, we'll use a subset
# Remove this line for full training
args.filter_n_atoms = 10  # Only molecules with <=10 atoms

print("Loading dataset...")
dataloaders, charge_scale = dataset.retrieve_dataloaders(args)

print(f"Training set size: {len(dataloaders['train'].dataset)}")
print(f"Validation set size: {len(dataloaders['valid'].dataset)}")
print(f"Test set size: {len(dataloaders['test'].dataset)}")

# Get dataset info
dataset_info = get_dataset_info('qm9', remove_h=args.remove_h)
print(f"\nAtom types: {dataset_info['atom_decoder']}")
print(f"Max nodes: {dataset_info['max_n_nodes']}")

### Examine a Sample

In [ ]:
# Get one batch
sample_batch = next(iter(dataloaders['train']))

print("Batch keys:", sample_batch.keys())
print(f"\nPositions shape: {sample_batch['positions'].shape}")
print(f"One-hot shape: {sample_batch['one_hot'].shape}")
print(f"Charges shape: {sample_batch['charges'].shape}")
print(f"Atom mask shape: {sample_batch['atom_mask'].shape}")

# Visualize first molecule
mol_idx = 0
positions = sample_batch['positions'][mol_idx].numpy()
atom_types = sample_batch['one_hot'][mol_idx].argmax(dim=-1).numpy()
atom_mask = sample_batch['atom_mask'][mol_idx, :, 0].numpy()

# Get actual atoms (not padded)
valid_atoms = atom_mask > 0
positions = positions[valid_atoms]
atom_types = atom_types[valid_atoms]

print(f"\nExample molecule:")
print(f"Number of atoms: {len(positions)}")
print(f"Atom types: {[dataset_info['atom_decoder'][t] for t in atom_types]}")

# Plot 3D structure
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

colors = ['red', 'gray', 'blue', 'darkred', 'yellow']
for atom_type in range(len(dataset_info['atom_decoder'])):
    mask = atom_types == atom_type
    if mask.any():
        ax.scatter(positions[mask, 0], positions[mask, 1], positions[mask, 2],
                  c=colors[atom_type], s=200, alpha=0.8,
                  label=dataset_info['atom_decoder'][atom_type])

ax.set_xlabel('X (Å)')
ax.set_ylabel('Y (Å)')
ax.set_zlabel('Z (Å)')
ax.legend()
ax.set_title('Sample Molecule from QM9')
plt.tight_layout()
plt.show()

## 2. Model Setup

We'll create an E3 equivariant diffusion model with EGNN dynamics.

In [ ]:
# Model hyperparameters
args.model = 'egnn_dynamics'
args.probabilistic_model = 'diffusion'

# EGNN architecture
args.nf = 128  # Hidden features
args.n_layers = 6  # Number of EGNN layers
args.attention = True
args.tanh = True
args.norm_constant = 1
args.inv_sublayers = 1
args.sin_embedding = False
args.normalization_factor = 1
args.aggregation_method = 'sum'

# Diffusion settings
args.diffusion_steps = 500
args.diffusion_noise_schedule = 'polynomial_2'
args.diffusion_noise_precision = 1e-5
args.diffusion_loss_type = 'l2'

# Normalization
args.normalize_factors = [1, 4, 1]  # [positions, features, charges]
args.conditioning = []
args.context_node_nf = 0

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
args.device = device
dtype = torch.float32

print("Creating model...")
model, nodes_dist, prop_dist = get_model(args, device, dataset_info, dataloaders['train'])
model = model.to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")
print(f"Model device: {next(model.parameters()).device}")

## 3. Training (Quick Demo)

For demonstration, we'll train for just a few epochs. For good results, train for 200-1000 epochs.

In [ ]:
# Training settings
args.n_epochs = 5  # Use 200+ for real training
args.lr = 2e-4
args.ema_decay = 0.999
args.clip_grad = True
args.clip_grad_norm = 1.0

# Create optimizer
optimizer = get_optim(args, model)

print(f"Training for {args.n_epochs} epochs...")
print("Note: For good results, train for 200+ epochs")

# Training loop
from equivariant_diffusion.utils import assert_mean_zero_with_mask, remove_mean_with_mask

train_losses = []
val_losses = []

for epoch in range(args.n_epochs):
    model.train()
    epoch_loss = 0
    n_batches = 0
    
    for batch_idx, data in enumerate(dataloaders['train']):
        # Move to device
        x = data['positions'].to(device, dtype)
        node_mask = data['atom_mask'].to(device, dtype)
        edge_mask = data['edge_mask'].to(device, dtype)
        one_hot = data['one_hot'].to(device, dtype)
        charges = data['charges'].to(device, dtype) if args.include_charges else torch.zeros(0)
        
        # Remove center of mass
        x = remove_mean_with_mask(x, node_mask)
        
        # Prepare features
        h = {'categorical': one_hot, 'integer': charges}
        
        # Forward pass
        optimizer.zero_grad()
        nll, reg_term, mean_abs_z = model(x, h, node_mask, edge_mask, context=None)
        loss = nll.mean()
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        if args.clip_grad:
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip_grad_norm)
        
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
        
        if batch_idx >= 10:  # Quick demo - only 10 batches per epoch
            break
    
    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)
    
    # Validation
    model.eval()
    val_loss = 0
    n_val_batches = 0
    
    with torch.no_grad():
        for batch_idx, data in enumerate(dataloaders['valid']):
            x = data['positions'].to(device, dtype)
            node_mask = data['atom_mask'].to(device, dtype)
            edge_mask = data['edge_mask'].to(device, dtype)
            one_hot = data['one_hot'].to(device, dtype)
            charges = data['charges'].to(device, dtype) if args.include_charges else torch.zeros(0)
            
            x = remove_mean_with_mask(x, node_mask)
            h = {'categorical': one_hot, 'integer': charges}
            
            nll, reg_term, mean_abs_z = model(x, h, node_mask, edge_mask, context=None)
            val_loss += nll.mean().item()
            n_val_batches += 1
            
            if batch_idx >= 5:  # Quick validation
                break
    
    avg_val_loss = val_loss / n_val_batches
    val_losses.append(avg_val_loss)
    
    print(f"Epoch {epoch+1}/{args.n_epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

print("\nTraining complete!")

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Sampling New Molecules

Now let's generate new molecules using the trained model!

In [ ]:
print("Generating molecules...")

model.eval()
n_samples = 5
n_nodes = 8  # Number of atoms

# Create masks
node_mask = torch.ones(n_samples, n_nodes, 1).to(device, dtype)
edge_mask = torch.ones(n_samples, n_nodes, n_nodes).to(device, dtype)

# Sample
with torch.no_grad():
    x, h = model.sample(n_samples, n_nodes, node_mask, edge_mask, context=None)

print(f"Generated {n_samples} molecules with {n_nodes} atoms each")
print(f"Positions shape: {x.shape}")
print(f"Features shape: {h['categorical'].shape}")

In [ ]:
# Visualize generated molecules
fig = plt.figure(figsize=(15, 3))

for i in range(min(5, n_samples)):
    ax = fig.add_subplot(1, 5, i+1, projection='3d')
    
    positions = x[i].cpu().numpy()
    atom_types = h['categorical'][i].argmax(dim=-1).cpu().numpy()
    
    colors = ['red', 'gray', 'blue', 'darkred', 'yellow']
    for atom_type in range(len(dataset_info['atom_decoder'])):
        mask = atom_types == atom_type
        if mask.any():
            ax.scatter(positions[mask, 0], positions[mask, 1], positions[mask, 2],
                      c=colors[atom_type], s=100, alpha=0.8)
    
    ax.set_title(f'Molecule {i+1}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    
    # Equal aspect ratio
    max_range = np.array([positions[:, 0].max()-positions[:, 0].min(),
                         positions[:, 1].max()-positions[:, 1].min(),
                         positions[:, 2].max()-positions[:, 2].min()]).max() / 2.0
    mid_x = (positions[:, 0].max()+positions[:, 0].min()) * 0.5
    mid_y = (positions[:, 1].max()+positions[:, 1].min()) * 0.5
    mid_z = (positions[:, 2].max()+positions[:, 2].min()) * 0.5
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)

plt.tight_layout()
plt.show()

## 5. Evaluate Molecule Quality

Let's check if generated molecules are chemically valid.

In [ ]:
from qm9.analyze import check_stability

# Check stability for generated molecules
stability_results = []

for i in range(n_samples):
    positions = x[i].cpu().numpy()
    atom_types = h['categorical'][i].argmax(dim=-1).cpu().numpy()
    
    # Get charges if available
    if 'integer' in h and h['integer'].numel() > 0:
        charges = h['integer'][i].cpu().numpy()
    else:
        charges = np.zeros(len(atom_types))
    
    # Check stability
    atom_stable, mol_stable, validity_dict = check_stability(
        positions, atom_types, charges, dataset_info
    )
    
    stability_results.append({
        'molecule': i,
        'atom_stable': atom_stable,
        'mol_stable': mol_stable,
        'validity': validity_dict
    })
    
    print(f"Molecule {i+1}:")
    print(f"  Atoms stable: {atom_stable}/{len(atom_types)}")
    print(f"  Molecule stable: {mol_stable}")
    print(f"  Composition: {[dataset_info['atom_decoder'][t] for t in atom_types]}")
    print()

# Overall statistics
total_stable = sum(r['mol_stable'] for r in stability_results)
print(f"\nOverall stability: {total_stable}/{n_samples} ({100*total_stable/n_samples:.1f}%)")
print("\nNote: These are results from a quick demo training.")
print("For high-quality molecules, train for 200+ epochs on full dataset.")

## 6. Save and Load Model

In [ ]:
# Save model
save_path = 'tutorial_model.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'args': args,
    'dataset_info': dataset_info,
}, save_path)

print(f"Model saved to {save_path}")

# Load model (example)
checkpoint = torch.load(save_path)
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

print("Model can be loaded from checkpoint")

## Summary

In this tutorial, you learned:

1. ✅ How to load and prepare QM9 dataset
2. ✅ How to create an E3 equivariant diffusion model
3. ✅ How to train the model (quick demo)
4. ✅ How to generate new molecules
5. ✅ How to evaluate molecule quality
6. ✅ How to save and load models

## Next Steps

- **Tutorial 2**: Conditional generation with properties
- **Tutorial 3**: Crystal generation
- **Tutorial 4**: Molecular descriptors and ASE databases
- **Tutorial 5**: Advanced evaluation and analysis

## For Production Use

```bash
# Train on full dataset with optimal hyperparameters
python main_qm9.py \
    --exp_name production_model \
    --n_epochs 1000 \
    --batch_size 64 \
    --lr 1e-4 \
    --nf 256 \
    --n_layers 9 \
    --diffusion_steps 1000 \
    --ema_decay 0.9999
```

Training time: ~2-3 days on V100 GPU